In [1]:
# ============================================================
# API KEY SETUP
# ============================================================

import os
import getpass

def get_key(name):
    key = os.environ.get(name, "")

    if not key:
        try:
            from google.colab import userdata
            key = userdata.get(name) or ""
        except Exception:
            key = ""

    if not key:
        key = getpass.getpass(
            f"Enter {name} (press Enter to skip): "
        ).strip()

    return key


GROQ_API_KEY = get_key("GROQ_API_KEY")
GEMINI_API_KEY = get_key("GEMINI_API_KEY")

print("Groq key provided:", bool(GROQ_API_KEY))
print("Gemini key provided:", bool(GEMINI_API_KEY))

if not GROQ_API_KEY and not GEMINI_API_KEY:
    print("No API key provided. Retrieval can still be tested.")

Enter GEMINI_API_KEY (press Enter to skip): ··········
Groq key provided: True
Gemini key provided: True


In [3]:
# ============================================================
# STEP 1-3 RECAP: CHUNK, STORE AND RETRIEVE
# ============================================================
!pip install -q langchain-text-splitters chromadb sentence-transformers rank_bm25 pandas

from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
from chromadb.utils import embedding_functions

documents = {
    "ai_overview.txt": """
Artificial Intelligence: An Overview

Artificial Intelligence, or AI, is the field of computer science
focused on building systems that can perform tasks which normally
require human intelligence. These tasks include understanding
language, recognizing images, making decisions, and learning.

Machine Learning is a subset of AI where systems learn patterns
from data instead of following hardcoded rules.

Deep Learning takes Machine Learning further by using neural
networks with many layers to discover complex patterns.

Retrieval-Augmented Generation, or RAG, combines a search system
with a language model. It retrieves relevant documents and passes
their information to the model to generate a grounded answer.
""",

    "embeddings_in_practice.txt": """
Embeddings and Vector Search in Practice

Modern search engines use embeddings rather than simple keyword
matching. An embedding model converts a query into a numerical
vector so the system can find documents with similar meanings.

This is useful in customer support systems where users may phrase
questions differently from the text in a help article.

Vector databases such as ChromaDB store embeddings and support
fast similarity searches across large document collections.
"""
}

splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=40,
    separators=["\n\n", "\n", ". ", " ", ""]
)

client = chromadb.Client()

hf_embedder = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# Recreate the collection to avoid duplicate IDs on reruns
try:
    client.delete_collection("rag_step4_collection")
except Exception:
    pass

collection = client.create_collection(
    name="rag_step4_collection",
    embedding_function=hf_embedder
)

for source, text in documents.items():
    chunks = splitter.split_text(text)

    collection.add(
        documents=chunks,
        metadatas=[
            {"source": source, "chunk_index": i}
            for i in range(len(chunks))
        ],
        ids=[
            f"{source}_chunk_{i}"
            for i in range(len(chunks))
        ]
    )

print("Knowledge base created!")
print("Total chunks:", collection.count())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 91.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Knowledge base created!
Total chunks: 7


In [4]:
# ============================================================
# RETRIEVAL FUNCTION
# ============================================================

def retrieve_context(
    query,
    n_results=4,
    similarity_threshold=0.25
):
    total_chunks = collection.count()

    if total_chunks == 0:
        return []

    results = collection.query(
        query_texts=[query],
        n_results=min(n_results, total_chunks),
        include=["documents", "metadatas", "distances"]
    )

    context_chunks = []

    for doc, meta, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        similarity = 1 - distance

        if similarity >= similarity_threshold:
            context_chunks.append({
                "text": doc,
                "source": meta["source"],
                "chunk_index": meta["chunk_index"],
                "similarity": round(similarity, 4)
            })

    return context_chunks


# Quick test
test_chunks = retrieve_context("What is RAG?")

print("Chunks retrieved:", len(test_chunks))

for chunk in test_chunks:
    print(
        chunk["similarity"],
        chunk["source"],
        chunk["chunk_index"]
    )
    print(chunk["text"])
    print("-" * 50)

Chunks retrieved: 1
0.5056 ai_overview.txt 4
Retrieval-Augmented Generation, or RAG, combines a search system
with a language model. It retrieves relevant documents and passes
their information to the model to generate a grounded answer.
--------------------------------------------------


In [5]:
# ============================================================
# GROUNDED PROMPT
# ============================================================

SYSTEM_PROMPT = (
    "You are a helpful assistant that answers questions using "
    "ONLY the provided context. If the answer is not supported "
    "by the context, reply exactly: "
    "\"I don't know based on the provided documents.\" "
    "Keep the answer short and clear. Mention the source file "
    "names used in your answer."
)


def build_user_prompt(query, context_chunks):
    context_text = "\n\n".join(
        f"[Source: {chunk['source']}, "
        f"chunk {chunk['chunk_index']}]\n{chunk['text']}"
        for chunk in context_chunks
    )

    return (
        f"Context:\n{context_text}\n\n"
        f"Question: {query}\n\n"
        "Answer:"
    )


sample_query = "How does RAG improve a language model?"
sample_chunks = retrieve_context(sample_query)

print(build_user_prompt(sample_query, sample_chunks))

Context:
[Source: ai_overview.txt, chunk 4]
Retrieval-Augmented Generation, or RAG, combines a search system
with a language model. It retrieves relevant documents and passes
their information to the model to generate a grounded answer.

[Source: ai_overview.txt, chunk 2]
language, recognizing images, making decisions, and learning.

Question: How does RAG improve a language model?

Answer:


In [8]:
# ============================================================
# GROQ LLM - UPDATED MODEL
# ============================================================

from groq import Groq

groq_client = Groq(api_key=GROQ_API_KEY)

# Check which models are available to your API key
available_models = {
    model.id for model in groq_client.models.list().data
}

print("Available text-generation models:")
for model_name in sorted(available_models):
    if "gpt-oss" in model_name:
        print("-", model_name)

# Select a supported model
if "openai/gpt-oss-120b" in available_models:
    GROQ_MODEL = "openai/gpt-oss-120b"
elif "openai/gpt-oss-20b" in available_models:
    GROQ_MODEL = "openai/gpt-oss-20b"
else:
    raise ValueError(
        "Neither recommended GPT-OSS model is available. "
        "Check the model IDs printed above."
    )

print("\nSelected model:", GROQ_MODEL)


def ask_groq(user_prompt):
    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.1,
        max_tokens=600
    )

    return response.choices[0].message.content


# Test the RAG pipeline
question = "How does RAG improve a language model?"
chunks = retrieve_context(question)

if chunks:
    prompt = build_user_prompt(question, chunks)
    answer = ask_groq(prompt)

    print("\nQuestion:", question)
    print("\nAI Answer:\n")
    print(answer)
else:
    print("I don't know based on the provided documents.")

Available text-generation models:
- openai/gpt-oss-120b
- openai/gpt-oss-20b
- openai/gpt-oss-safeguard-20b

Selected model: openai/gpt-oss-120b

Question: How does RAG improve a language model?

AI Answer:

RAG enhances a language model by retrieving relevant documents and feeding their information into the model, allowing it to generate answers that are grounded in external sources. (Source: ai_overview.txt)


Call Google Gemini

In [9]:
# ============================================================
# GOOGLE GEMINI LLM
# ============================================================

from google import genai
from google.genai import types

GEMINI_MODELS = [
    "gemini-2.5-flash",
    "gemini-2.5-flash-lite",
    "gemini-3-flash-preview"
]


def ask_gemini(user_prompt):
    if not GEMINI_API_KEY:
        return "[Gemini skipped: no API key provided]"

    gemini_client = genai.Client(api_key=GEMINI_API_KEY)
    last_error = None

    for model_name in GEMINI_MODELS:
        try:
            response = gemini_client.models.generate_content(
                model=model_name,
                contents=user_prompt,
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM_PROMPT,
                    temperature=0.1,
                    max_output_tokens=1500
                )
            )

            if response.text:
                return response.text.strip()

        except Exception as e:
            last_error = e

    return f"[Gemini error: {last_error}]"


question = "How does RAG improve a language model?"
chunks = retrieve_context(question)

if chunks:
    prompt = build_user_prompt(question, chunks)
    print(ask_gemini(prompt))
else:
    print("I don't know based on the provided documents.")

[Gemini error: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key not valid. Please pass a valid API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API key not valid. Please pass a valid API key.'}]}}]


Combine everything into rag_answer()

In [10]:
# ============================================================
# COMPLETE RAG PIPELINE
# ============================================================

def rag_answer(query, provider="groq", show_sources=True):
    context_chunks = retrieve_context(query)

    # Don't call the LLM if retrieval finds nothing relevant
    if not context_chunks:
        return (
            "I don't know based on the provided documents. "
            "(No relevant chunks were retrieved.)"
        )

    user_prompt = build_user_prompt(query, context_chunks)

    if provider.lower() == "gemini":
        answer = ask_gemini(user_prompt)
    elif provider.lower() == "groq":
        answer = ask_groq(user_prompt)
    else:
        return "Invalid provider. Choose 'groq' or 'gemini'."

    if show_sources:
        sources = sorted({
            f"{chunk['source']} #{chunk['chunk_index']}"
            for chunk in context_chunks
        })

        answer += "\n\nRetrieved from: " + ", ".join(sources)

    return answer


# Automatically select an available provider
if GROQ_API_KEY:
    default_provider = "groq"
elif GEMINI_API_KEY:
    default_provider = "gemini"
else:
    default_provider = None

print("Selected provider:", default_provider or "None")

Selected provider: groq


In [11]:
# TEST 1: Simple in-scope question

q1 = "What is deep learning?"

print("Question:", q1)
print()

if default_provider:
    print(rag_answer(q1, provider=default_provider))
else:
    chunks = retrieve_context(q1)
    print("Retrieved context:", chunks)
    print("Add an API key to generate an LLM answer.")

Question: What is deep learning?

Deep learning is an advanced form of machine learning that uses neural networks with many layers to discover complex patterns. (Source: ai_overview.txt, chunk 3)

Retrieved from: ai_overview.txt #0, ai_overview.txt #1, ai_overview.txt #2, ai_overview.txt #3


In [12]:
# TEST 2: Question requiring multiple concepts

q2 = "How do embeddings and vector databases help RAG find useful information?"

print("Question:", q2)
print()

if default_provider:
    print(rag_answer(q2, provider=default_provider))
else:
    print(retrieve_context(q2))

Question: How do embeddings and vector databases help RAG find useful information?

Embeddings turn a query (and documents) into numerical vectors, and a vector database such as ChromaDB stores those vectors and can quickly find documents whose vectors are most similar to the query. RAG then uses this similarity search to retrieve the most relevant documents and feeds them to the language model for a grounded answer【embeddings_in_practice.txt, chunk 0】【embeddings_in_practice.txt, chunk 1】【ai_overview.txt, chunk 4】.

Retrieved from: ai_overview.txt #3, ai_overview.txt #4, embeddings_in_practice.txt #0, embeddings_in_practice.txt #1


In [13]:
# TEST 3: Out-of-scope question

q3 = "What is the best recipe for chocolate cake?"

print("Question:", q3)
print()

if default_provider:
    print(rag_answer(q3, provider=default_provider))
else:
    chunks = retrieve_context(q3)

    if not chunks:
        print("I don't know based on the provided documents.")
    else:
        print("Potentially relevant chunks:", chunks)

Question: What is the best recipe for chocolate cake?

I don't know based on the provided documents. (No relevant chunks were retrieved.)


Compare Groq and Gemini

In [14]:
compare_q = "Why is RAG more reliable than asking a language model directly?"

compare_chunks = retrieve_context(compare_q)

if not compare_chunks:
    print("I don't know based on the provided documents.")
else:
    compare_prompt = build_user_prompt(
        compare_q,
        compare_chunks
    )

    print("=" * 25, "GROQ", "=" * 25)

    if GROQ_API_KEY:
        print(ask_groq(compare_prompt))
    else:
        print("Groq skipped: no API key.")

    print("\n" + "=" * 24, "GEMINI", "=" * 24)

    if GEMINI_API_KEY:
        print(ask_gemini(compare_prompt))
    else:
        print("Gemini skipped: no API key.")

========================= GROQ =========================
RAG is more reliable because it first pulls in relevant documents and then feeds that factual information to the language model, producing a grounded answer rather than relying solely on the model’s internal knowledge. (Source: ai_overview.txt)

======================== GEMINI ========================
[Gemini error: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key not valid. Please pass a valid API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API key not valid. Please pass a valid API key.'}]}}]


In [ ]:
while True:
    user_question = input(
        "Ask a question (or press Enter to quit): "
    ).strip()

    if not user_question:
        print("Goodbye!")
        break

    if default_provider:
        print(
            "\n" + rag_answer(
                user_question,
                provider=default_provider
            ) + "\n"
        )
    else:
        chunks = retrieve_context(user_question)

        if not chunks:
            print("I don't know based on the provided documents.\n")
        else:
            print("Retrieved context:\n")

            for chunk in chunks:
                print(chunk["text"])
                print("Source:", chunk["source"])
                print("-" * 50)

In [ ]:
# Add a document to the existing collection

third_document = """
Python Programming

Python is a high-level programming language.
NumPy supports numerical computing.
Pandas supports data analysis and data manipulation.
Scikit-learn provides machine learning algorithms.
"""

source = "python_programming.txt"
third_chunks = splitter.split_text(third_document)

collection.add(
    documents=third_chunks,
    metadatas=[
        {"source": source, "chunk_index": i}
        for i in range(len(third_chunks))
    ],
    ids=[
        f"{source}_chunk_{i}"
        for i in range(len(third_chunks))
    ]
)

print("New chunks added:", len(third_chunks))
print("Total chunks:", collection.count())

question = "Which Python library supports data analysis?"

if default_provider:
    print(rag_answer(question, provider=default_provider))
else:
    print(retrieve_context(question))